# Assignment 6 — Solution

**Course:** COMP 7712 - Algorithms

**Due Date:** 4/21/2026, 9:00 AM

**Topics:** Divide and Conquer, Master Theorem, Karatsuba's Trick, Peak Finding

---

## Part 1 — Integer Multiplication

Throughout Part 1, $n$ is a power of 2 and each $n$-digit number is split as

$$x = a \cdot 10^{n/2} + b, \qquad y = c \cdot 10^{n/2} + d,$$

so that

$$x \cdot y = ac \cdot 10^{n} + (ad + bc) \cdot 10^{n/2} + bd.$$

Pseudocode:
```python
def multiply(x, y, n):
    # 1. Base case: a single-digit multiplication is one primitive op.
    if n == 1:
        return x * y

    # 2. Divide: split each number into high and low halves of n/2 digits.
    half = n / 2
    a = x // 10^half        # high half of x
    b = x  mod 10^half      # low  half of x
    c = y // 10^half        # high half of y
    d = y  mod 10^half      # low  half of y

    # 3. Conquer: four recursive multiplications on n/2-digit numbers.
    ac = multiply(a, c, half)
    ad = multiply(a, d, half)
    bc = multiply(b, c, half)
    bd = multiply(b, d, half)

    # 4. Combine: assemble the product using shifts (powers of 10) and adds.
    return ac * 10^n  +  (ad + bc) * 10^half  +  bd
```

$$T(n) = \Theta(n) + 4T({n \over 2}) \in \Theta(n^2)$$


### Problem 1.1 — Hand computation

With $x = 1234$, $y = 5678$, $n=4$, $n/2=2$.

**Solution:**

**(a)** Split each 4-digit number into its high and low 2-digit halves:
$$a = 12,\quad b = 34,\quad c = 56,\quad d = 78.$$

**(b)** Four products (computed by hand):

- $ac = 12 \cdot 56 = 672$
- $ad = 12 \cdot 78 = 936$
- $bc = 34 \cdot 56 = 1904$
- $bd = 34 \cdot 78 = 2652$

**(c)** Assemble:
$$x\cdot y = ac\cdot 10^{4} + (ad+bc)\cdot 10^{2} + bd = 672\cdot 10000 + (936+1904)\cdot 100 + 2652.$$
$$= 6\,720\,000 + 2\,840 \cdot 100 + 2652 = 6\,720\,000 + 284\,000 + 2\,652 = 7\,006\,652.$$

Direct check: $1234\cdot 5678 = 7{,}006{,}652$. ✓

### Problem 1.2 — Recurrence for the naive D&C algorithm

**Solution:**

**(a)** Four recursive multiplications on $n/2$-digit numbers, plus $\Theta(n)$ work to add and shift:
$$T(n) = 4\,T(n/2) + \Theta(n).$$

**(b)** Master-Theorem form $T(n) = a\,T(n/b) + \Theta(n^{d})$ with $a = 4,\ b = 2,\ d = 1$.
$\log_b a = \log_2 4 = 2 > 1 = d$, so **Case 1** applies:
$$T(n) = \Theta\!\left(n^{\log_2 4}\right) = \Theta(n^{2}).$$

**(c)** This matches the $\Theta(n^2)$ grade-school algorithm exactly — doing four recursive multiplies does not buy us anything.

### Problem 1.3 — The algebraic trick

**Solution:**

**(a)** Expand
$$(a+b)(c+d) = ac + ad + bc + bd.$$
Rearranging,
$$ad + bc = (a+b)(c+d) - ac - bd.$$

**(b)** The three half-size multiplications are
$$P_1 = ac,\qquad P_2 = bd,\qquad P_3 = (a+b)(c+d).$$
Then the middle coefficient is recovered without any extra multiply:
$$ad + bc = P_3 - P_1 - P_2,$$
and
$$x\cdot y = P_1 \cdot 10^{n} + (P_3 - P_1 - P_2) \cdot 10^{n/2} + P_2.$$
Only additions, subtractions, and (base-10) shifts are used to combine the three products, so we've gone from four half-size multiplications to **three**.

### Problem 1.4 — Karatsuba's recurrence and the Master Theorem

**Solution:**

**(a)** Three recursive multiplies on $n/2$-digit inputs, plus $\Theta(n)$ additions and shifts:
$$T(n) = 3\,T(n/2) + \Theta(n).$$

**(b)** $a = 3,\ b = 2,\ d = 1$. $\log_b a = \log_2 3 \approx 1.585 > 1 = d$, so **Case 1** applies:
$$T(n) = \Theta\!\left(n^{\log_2 3}\right) \approx \Theta(n^{1.585}).$$

### Problem 1.5 — Which knob did we change?

**Solution:**

**(a)** Comparing the two recurrences:

| knob | naive (1.2) | Karatsuba (1.4) |
|------|-------------|-----------------|
| $a$  | 4           | **3**           |
| $b$  | 2           | 2               |
| $d$  | 1           | 1               |

Only $a$ changed — from 4 down to 3 — by using the algebraic identity to save one multiplication per level. The split factor $b$ and the combine cost $d$ are unchanged.

**(b)** Estimating $\log_2 3$.
Given $2^{1.5}\approx 2.83$ and $2^{1.6}\approx 3.03$, we need the exponent $t$ with $2^{t} = 3$. Linearly interpolating between the two anchor points,
$$t \approx 1.5 + \frac{3 - 2.83}{3.03 - 2.83}\cdot 0.1 = 1.5 + \frac{0.17}{0.20}\cdot 0.1 \approx 1.5 + 0.085 \approx 1.585.$$
So $\log_2 3 \approx 1.58$–$1.59$.

**(c)** At $n = 1024 = 2^{10}$:
$$n^{\log_2 3} = (2^{10})^{\log_2 3} = 2^{10\log_2 3} = 3^{10} = 59\,049,$$
$$n^{2} = 2^{20} = 1\,048\,576.$$
Ratio:
$$\frac{n^{2}}{n^{\log_2 3}} = \frac{1\,048\,576}{59\,049} \approx 17.76.$$
So at $n=1024$ Karatsuba does on the order of **17–18×** less work than the grade-school/naive algorithm (ignoring constant factors).

---

## Part 2 — Peak Finding


Given a list `L` of length $n$, we say index $i$ is a **peak** if
- $L[i] \geq L[i-1]$ and $L[i] \geq L[i+1]$ for an interior index $0 < i < n-1$,
- $L[0] \geq L[1]$ if $i = 0$, and
- $L[n-1] \geq L[n-2]$ if $i = n-1$.

The goal of this part is to design an algorithm that finds **any one** peak — not all of them, not the largest — and to analyze its running time.


### Problem 2.1 — Hand exploration

`L = [1, 3, 7, 2, 6, 4, 5]`

**Solution:**

**(a)** Check each index $i$ against the definition:

| $i$ | $L[i]$ | left nbr | right nbr | peak? |
|-----|--------|----------|-----------|-------|
| 0   | 1      | —        | 3         | $1\ge 3$? No |
| 1   | 3      | 1        | 7         | $3\ge 7$? No |
| 2   | 7      | 3        | 2         | $7\ge 3$ and $7\ge 2$ → **peak** |
| 3   | 2      | 7        | 6         | $2\ge 7$? No |
| 4   | 6      | 2        | 4         | $6\ge 2$ and $6\ge 4$ → **peak** |
| 5   | 4      | 6        | 5         | $4\ge 6$? No |
| 6   | 5      | 4        | —         | $5\ge 4$ → **peak** |

So the peak indices are $\{2,\ 4,\ 6\}$.

**(b)** The endpoints only have one neighbor, so we only compare against that single neighbor (index 0 compares with index 1; index $n-1$ compares with index $n-2$).

### Problem 2.2 — Brute-force implementation

In [9]:
def peak_brute(L):
    n = len(L)
    if n == 1:
        return 0
    if L[0] >= L[1]:
        return 0
    if L[n-1] >= L[n-2]:
        return n - 1
    i = 1
    while i < n - 1:
        print("i =", i)
        if L[i] >= L[i-1] and L[i] >= L[i+1]:
            return i
        i += 1
    # A finite list of real numbers always contains a peak (the maximum is one),
    # so we never fall through here, but keep a sentinel for safety.
    return -1

# Quick sanity checks — feel free to add your own.
peak_brute([1, 3, 7, 2, 6, 4, 3]) 

i = 1
i = 2


2

**Worst-case running time:** $\Theta(n)$. Each index is inspected at most once with $O(1)$ work. A worst case is `L = [0, 1, 2, ..., n-2, -1]` (strictly increasing except for a drop at the end): neither endpoint is a peak, so the two endpoint shortcuts don't fire, and the loop must walk from $i=1$ all the way to $i=n-2$ before finding the peak.

### Problem 2.3 — The invariant (why D&C works)

**Proof.**
Suppose $L[m-1] > L[m]$. We show that the sublist $L[0\,..\,m-1]$ contains an index that is a peak *of the original list*. Note: to be a peak of the original list, an interior index $i$ must satisfy $L[i]\ge L[i-1]$ and $L[i]\ge L[i+1]$ using the original neighbors; the only index in $L[0\,..\,m-1]$ whose "right" neighbor differs between the sublist and the full list is $m-1$, whose full-list right neighbor is $L[m]$.

Now walk leftward from $m-1$: starting at $j=m-1$, while $j>0$ and $L[j-1]>L[j]$, decrement $j$. The walk must terminate, and it terminates in one of two ways. (i) We hit an interior index $j$ with $L[j-1]\le L[j]$; by construction of the walk we also have $L[j]\ge L[j+1]$ (either $j+1=m$ and $L[j]\ge L[m]$ is how the walk advanced, or $L[j]>L[j+1]$ from the previous strict step), so $j$ is a peak of the full list. (ii) We reach $j=0$; then by construction $L[0]\ge L[1]$, so index $0$ is a peak of the full list. In both cases the peak lies in $L[0\,..\,m-1]$, as claimed. $\blacksquare$

---

Example: 
```
index:   0   1   2   3   4   5   6   7   8   9
L:      [1,  3,  5,  4,  2,  8,  6,  7,  9,  3]
```
i = 3

### Problem 2.4 — Predict the running time

**Prediction:**

**(a)** The invariant (and its left-right symmetric twin) lets us discard half the list after a single comparison, and we recurse into exactly one half:
$$T(n) = T(n/2) + \Theta(1).$$

**(b)** Master-Theorem form with $a=1,\ b=2,\ d=0$. Since $\log_2 1 = 0 = d$, **Case 2** applies:
$$T(n) = \Theta(n^{d}\log n) = \Theta(\log n).$$

### Problem 2.5 — Divide-and-conquer implementation

We use a helper that tracks the *original* index bounds `lo, hi` so the returned index refers to the original list. At each step we look at the midpoint and the element to its right: whichever side is "uphill" from the midpoint must contain a peak (by the claim from 2.3 and its mirror).

In [15]:
def peak_dc(L):
    def helper(lo, hi):
        print("[", lo, " ,", hi, "]")
        # Returns a peak index of L, chosen from the window L[lo .. hi].
        if lo == hi:
            return lo
        m = (lo + hi) // 2
        if L[m] < L[m + 1]:
            return helper(m + 1, hi)
        else:
            return helper(lo, m)
            
    return helper(0, len(L) - 1)

# Sanity checks.
peak_dc([1, 3, 7, 2, 6, 4, 5])


[ 0  , 6 ]
[ 4  , 6 ]
[ 6  , 6 ]


6

### Problem 2.6 — Recurrence and Master Theorem

**Solution:**

**(a)** Each call does $O(1)$ work (one comparison `L[m+1] > L[m]`, a midpoint computation) and makes **one** recursive call on a window of size at most $\lceil n/2\rceil$. No list slicing is performed — we pass index bounds, so there is no hidden linear cost:
$$T(n) = T(n/2) + \Theta(1).$$

**(b)** $a = 1,\ b = 2,\ d = 0$. $\log_2 1 = 0 = d$, so **Case 2** applies:
$$T(n) = \Theta(\log n).$$

**(c)** This matches the prediction in Problem 2.4 exactly. The key structural feature is that we only recurse into **one** half (so $a = 1$, not $2$) and the non-recursive work is $O(1)$ (so $d = 0$). These two together are what give $\log n$.

### Problem 2.7 — Empirical verification

A caveat before running: on *uniformly random* lists, peaks are very common (a random interior index is a peak with probability $\approx 1/3$), so `peak_brute` will almost always terminate after inspecting only a handful of indices. This hides the $\Theta(n)$ worst-case behavior. To see the asymptotic ratio predicted by the Master Theorem, we also time on a true worst case for `peak_brute`: `L = [0, 1, ..., n-2, -1]` (strictly increasing except for a drop at the very end). On this input, `peak_brute` must scan to index $n-2$, while `peak_dc` is unaffected.

In [ ]:
import time, random
random.seed(7712)

# Bump Python's recursion limit so peak_dc can comfortably handle n = 10**6.
# The actual depth is ~log2(n) ≈ 20, well within limits, but set this to be safe.
import sys
sys.setrecursionlimit(100000)

def time_once(fn, L, reps):
    # Time `reps` calls and return the average wall-clock seconds per call.
    start = time.perf_counter()
    for _ in range(reps):
        fn(L)
    return (time.perf_counter() - start) / reps

sizes = [10**3, 10**4, 10**5, 10**6]

print("=== Random lists (typical case: peak_brute short-circuits fast) ===")
print(f"{'n':>10} | {'t_brute (s)':>14} | {'t_dc (s)':>12} | {'ratio':>10}")
for n in sizes:
    L = [random.randint(0, 10**9) for _ in range(n)]
    reps_b = 200 if n <= 10**4 else 50
    reps_d = 2000 if n <= 10**5 else 500
    tb = time_once(peak_brute, L, reps_b)
    td = time_once(peak_dc,    L, reps_d)
    print(f"{n:>10} | {tb:>14.8f} | {td:>12.8f} | {tb/td:>10.2f}")

print()
print("=== Worst case for peak_brute: L = [0, 1, ..., n-2, -1] ===")
print(f"{'n':>10} | {'t_brute (s)':>14} | {'t_dc (s)':>12} | {'ratio':>10}")
for n in sizes:
    L = list(range(n))
    L[-1] = -1  # force the peak to be at index n-2, not at either endpoint
    reps_b = 5 if n >= 10**5 else 50
    reps_d = 2000 if n <= 10**5 else 500
    tb = time_once(peak_brute, L, reps_b)
    td = time_once(peak_dc,    L, reps_d)
    print(f"{n:>10} | {tb:>14.8f} | {td:>12.8f} | {tb/td:>10.2f}")

**Interpretation:**

On *random* inputs the ratio `t_brute / t_dc` stays small and roughly flat in $n$ — in fact `peak_brute` is faster than `peak_dc` here, because a random list has lots of peaks and `peak_brute` almost always finds one in the first few indices, giving it effectively $O(1)$ behavior on random data. That does **not** contradict the Master-Theorem prediction; the prediction is about *worst-case* running time.

On the *worst-case* input `[0, 1, ..., n-2, -1]`, the ratio grows roughly like $n/\log_2 n$. Going from $n=10^3$ to $n=10^6$, $n$ grows by $1000\times$ while $\log_2 n$ grows from $\approx 10$ to $\approx 20$, so we'd expect the ratio to grow by roughly $500\times$. The measured ratios track this scaling, which is exactly what we'd predict from $\Theta(n)$ vs $\Theta(\log n)$ — matching Problems 2.4 and 2.6.

---

## Closing Reflection

**Solution:**

**(a)** Master-Theorem form $T(n) = a\,T(n/b) + \Theta(n^{d})$:

| problem          | $a$ | $b$ | $d$ | complexity                   |
|------------------|-----|-----|-----|------------------------------|
| Karatsuba        | 3   | 2   | 1   | $\Theta(n^{\log_2 3})$        |
| Peak finding D&C | 1   | 2   | 0   | $\Theta(\log n)$              |

**(b)** For Karatsuba, the essential lever was **$a$**: we changed $a$ from 4 (naive D&C) to 3 using the algebraic identity, while leaving $b$ and $d$ untouched. Because $d < \log_b a$ (Case 1 of the Master Theorem), the complexity is dictated by $a$, and shrinking $a$ shrinks the exponent $\log_b a$ from 2 to $\log_2 3 \approx 1.585$.

For peak finding, the essential lever is that **$a = 1$**, combined with $d = 0$: recursing into only one half instead of two is what collapses the tree from a full binary tree of size $\Theta(n)$ down to a single path of length $\Theta(\log n)$.

**(c)** The structure of a problem usually pins down $b$ (how much smaller the subproblem is — often $b=2$) and $d$ (how expensive the unavoidable combine work is). The "design space" is therefore most often about $a$: can you share or reuse work across subproblems to make fewer recursive calls (Karatsuba: 4 → 3), or can you prove an invariant that lets you *skip* entire subproblems (peak finding, binary search: effectively $a = 1$)? Cleverness typically lives in shrinking $a$; $b$ and $d$ are the parts the problem hands you.